# AIC HCMC 2025 — VietASR transcript

**Before running: `Kernel → Restart & Clear Outputs`**

Extracts a Vietnamese transcript from a competition video using
[VietASR](https://github.com/zzasdf/VietASR) (icefall zipformer, checkpoint
`zzasdf/viet_iter3_pseudo_label`). Audio is extracted to a 16 kHz mono WAV
with ffmpeg, segmented by Silero VAD (clean word boundaries, silence skipped),
then decoded segment-by-segment with `modified_beam_search` for maximum quality
(the zipformer OOMs on GPU if the whole video is one long sequence).

- Video: `L27_V010.mp4` (Kaggle dataset `tommr211/aic-batch1-videos`)
- Output: `/kaggle/working/L27_V010_transcript.txt`

**Internet must be enabled** (pip installs + HuggingFace download). GPU recommended.

## 1. Install dependencies (k2 / kaldifeat / lhotse / icefall)

In [ ]:
import os, subprocess, sys, time

PYPI = "https://pypi.org/simple"

def pip(*args):
    for attempt in range(3):
        try:
            subprocess.check_call(
                [sys.executable, "-m", "pip", "install", "-q",
                 "--index-url", PYPI, "--retries", "5", *args]
            )
            return
        except subprocess.CalledProcessError:
            if attempt == 2:
                raise
            time.sleep(10)

import torch
print("Python:", sys.version.split()[0])
print("torch:", torch.__version__, "| cuda:", torch.version.cuda, "| gpu:", torch.cuda.is_available())

# Quick connectivity check -> tells you clearly if Internet is OFF
try:
    import urllib.request
    urllib.request.urlopen(PYPI, timeout=10)
except Exception as e:
    raise SystemExit(
        "Cannot reach PyPI (%s).\n"
        "=> Enable Internet: Kaggle right panel -> Session options -> Internet: ON,"
        " then re-run this cell." % e
    )

# Pure-python / simple packages first (kaldialign + pypinyin are imported by icefall, silero-vad for VAD)
pip("lhotse", "sentencepiece", "huggingface_hub", "kaldialign", "pypinyin", "silero-vad")

K2_CUDA = "https://k2-fsa.github.io/k2/cuda.html"

# kaldifeat is NOT needed (pretrained.py is patched to torchaudio fbank).
# k2 is a pre-release on its own index -> plain `pip install k2` grabs the
# UNRELATED PyPI "k2" package (no _k2 module). Pin the exact wheel instead.
import re, urllib.request

if not torch.cuda.is_available():
    print("NOTE: CPU only (no GPU accelerator) -> decoding works but is slow;")
    print("      enable a GPU accelerator in Kaggle Settings for speed.")

def k2_pin():
    py = f"cp{sys.version_info.major}{sys.version_info.minor}"
    t = torch.__version__.split("+")[0]
    cuda = torch.version.cuda
    if cuda:
        index, key = K2_CUDA, f"+cuda{cuda}.torch{t}"
    else:
        index, key = "https://k2-fsa.github.io/k2/cpu.html", f"+cpu.torch{t}"
    html = urllib.request.urlopen(index, timeout=30).read().decode()
    for m in re.finditer(r"k2-(\S+?)-" + py, html):
        if key in m.group(1):
            return m.group(1), index
    return None, None

ver, idx = k2_pin()
if ver is None:
    raise SystemExit(
        f"No k2 wheel for torch {torch.__version__} (cuda {torch.version.cuda}, "
        f"python {sys.version_info.major}.{sys.version_info.minor}).\n"
        "=> Pin torch in the fallback cell below, then restart."
    )
subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "-q",
     "--index-url", PYPI, f"k2=={ver}", "-f", idx]
)
import k2
print("k2 OK:", ver)

# icefall is a collection of scripts (not pip-installed here) — VietASR imports it
if not os.path.isdir("icefall"):
    subprocess.check_call(["git", "clone", "-q", "https://github.com/k2-fsa/icefall"])
os.environ["PYTHONPATH"] = os.path.abspath("icefall") + ":" + os.environ.get("PYTHONPATH", "")
print("icefall ready at", os.path.abspath("icefall"))

### Fallback (only if the cell above failed)

If there is no CUDA wheel for Kaggle's current torch, pin torch to a version that
has wheels. Run the command below **in a terminal cell**, then
`Kernel → Restart` and run all cells again:

```bash
pip install -q torch==2.5.1 torchaudio==2.5.1 --index-url https://download.pytorch.org/whl/cu121
```

## 2. Fetch VietASR + model checkpoint

In [ ]:
import os, subprocess
from huggingface_hub import hf_hub_download

if not os.path.isdir("VietASR"):
    subprocess.check_call(["git", "clone", "-q", "https://github.com/zzasdf/VietASR"])

# pretrained.py uses kaldifeat (no CUDA wheel for this torch) ->
# patch it to use torchaudio's Kaldi-compatible fbank instead (identical features).
from pathlib import Path
p = Path("VietASR/ASR/zipformer/pretrained.py")
src = p.read_text()
old = '''    logging.info("Constructing Fbank computer")
    opts = kaldifeat.FbankOptions()
    opts.device = device
    opts.frame_opts.dither = 0
    opts.frame_opts.snip_edges = False
    opts.frame_opts.samp_freq = params.sample_rate
    opts.mel_opts.num_bins = params.feature_dim
    opts.mel_opts.high_freq = -400

    fbank = kaldifeat.Fbank(opts)

    logging.info(f"Reading sound files: {params.sound_files}")
    waves = read_sound_files(
        filenames=params.sound_files, expected_sample_rate=params.sample_rate
    )
    waves = [w.to(device) for w in waves]

    logging.info("Decoding started")
    features = fbank(waves)'''
new = '''    logging.info(f"Reading sound files: {params.sound_files}")
    waves = read_sound_files(
        filenames=params.sound_files, expected_sample_rate=params.sample_rate
    )

    logging.info("Decoding started")
    features = [
        torchaudio.compliance.kaldi.fbank(
            w.unsqueeze(0),
            num_mel_bins=params.feature_dim,
            sample_frequency=params.sample_rate,
            dither=0.0,
            snip_edges=False,
            high_freq=-400,
        ).to(device)
        for w in waves
    ]'''
if "import kaldifeat" not in src:
    print("pretrained.py already patched, skipping")
else:
    src = src.replace("import kaldifeat\n", "")
    assert old in src, "fbank block not found in pretrained.py"
    p.write_text(src.replace(old, new))
    print("patched pretrained.py -> torchaudio fbank (no kaldifeat)")

REPO = "zzasdf/viet_iter3_pseudo_label"
CKPT = hf_hub_download(REPO, "exp/epoch-12.pt")
TOKENS = hf_hub_download(REPO, "data/Vietnam_bpe_2000_new/tokens.txt")
print("checkpoint:", CKPT)
print("tokens:", TOKENS)

## 3. Extract audio + VAD speech segments

In [ ]:
VIDEO = "/kaggle/input/datasets/tommr211/aic-batch1-videos/Videos_L27_a/video/L27_V010.mp4"
WAV = "/kaggle/working/L27_V010.wav"
SEG_DIR = "/kaggle/working/segs"

subprocess.run(
    ["ffmpeg", "-y", "-i", VIDEO, "-vn", "-ac", "1", "-ar", "16000", "-c:a", "pcm_s16le", WAV],
    check=True, capture_output=True,
)
print("wav written:", WAV)

# VAD: split at silence so no word is cut mid-way, and cap long speech at 30s
# (the zipformer OOMs if a segment is too long).
import os
import torch
import soundfile as sf
from silero_vad import load_silero_vad, get_speech_timestamps

wav, sr = sf.read(WAV, dtype="float32")
model = load_silero_vad()
speech = get_speech_timestamps(
    torch.from_numpy(wav), model, sampling_rate=sr,
    return_seconds=True,
    max_speech_duration_s=30,
    min_silence_duration_ms=200,
    speech_pad_ms=100,
)

os.makedirs(SEG_DIR, exist_ok=True)
SEGS = []
for i, ts in enumerate(speech):
    s = max(0, int(ts["start"] * sr))
    e = min(len(wav), int(ts["end"] * sr))
    if e - s < sr * 0.3:
        continue
    p = f"{SEG_DIR}/seg_{i:03d}.wav"
    sf.write(p, wav[s:e], sr)
    SEGS.append(p)
print("speech segments:", len(SEGS))

## 4. Decode each segment (modified_beam_search)

In [ ]:
import re
VIETASR_DIR = os.path.abspath("VietASR/ASR/zipformer")
env = os.environ.copy()
env["PYTHONPATH"] = os.path.abspath("icefall") + ":" + env.get("PYTHONPATH", "")

transcripts = []
for i, seg in enumerate(SEGS):
    name = os.path.basename(seg)
    cmd = [sys.executable, "pretrained.py",
           "--checkpoint", CKPT, "--tokens", TOKENS,
           "--method", "modified_beam_search", seg]
    proc = subprocess.run(cmd, cwd=VIETASR_DIR, env=env, capture_output=True, text=True)
    log = proc.stdout + "\n" + proc.stderr
    if proc.returncode != 0:
        print("--- segment failed, log tail ---")
        print(log[-3000:])
        raise SystemExit(f"pretrained.py failed on {name} (exit {proc.returncode})")
    m = re.search(re.escape(name) + r":\n(.*?)\n", log)
    t = m.group(1).strip() if m else ""
    if t:
        transcripts.append(t)
    print(f"[{i+1}/{len(SEGS)}] {name}: {t[:120]}")

FULL = " ".join(transcripts)

## 5. Save transcript

In [ ]:
print("TRANSCRIPT:\n" + FULL)

out_txt = "/kaggle/working/L27_V010_transcript.txt"
with open(out_txt, "w") as f:
    f.write(FULL + "\n")
print("saved:", out_txt)